In [ ]:
"""
This file evaluates both the base model and the fine-tuned LoRA adapter on a set of 
test examples, checking that the model's output is valid JSON conforming to the 
expected schema, and comparing the performance of the base and fine-tuned models on the 
examples.

"""

In [1]:
# imports
import sys, pdb
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel
from dataclasses import dataclass, field
from dataset_generator import SYSTEM_INSTRUCTION
from typing import Dict, List
from evaluation import evaluate_model, print_evaluation_report

W1016 03:00:45.051000 11868 Lib\site-packages\torch\utils\flop_counter.py:113] triton not found; flop counting will not work for triton kernels


In [2]:
# config values - Se copia tal cual el notebook de entrenamiento.
@dataclass
class Config:
    """Every tuneable knob in one place."""

    # Model
    model_name: str = "google/gemma-2b"
    output_dir: str = "./gemma_lora_json_adapter"

    # LoRA
    lora_r: int = 16
    lora_alpha: int = 32
    lora_dropout: float = 0.05
    lora_target_modules: List[str] = field(
        default_factory=lambda: ["q_proj", "k_proj", "v_proj", "o_proj"]
    )

    # Quantization (QLoRA)
    use_4bit: bool = True
    bnb_4bit_compute_dtype: str = "bfloat16"
    bnb_4bit_quant_type: str = "nf4"
    use_nested_quant: bool = True

    # Training
    num_train_epochs: int = 3
    max_steps: int = 300          # set <=0 to use epochs instead
    per_device_train_batch_size: int = 2
    per_device_eval_batch_size: int = 2
    gradient_accumulation_steps: int = 8
    learning_rate: float = 2e-4
    weight_decay: float = 0.01
    warmup_ratio: float = 0.03
    max_seq_length: int = 256

    # Generation
    max_new_tokens: int = 120
    temperature: float = 1.0      # ignored when do_sample=False
    top_p: float = 1.0            # ignored when do_sample=False
    do_sample: bool = False       # greedy decoding = deterministic, clean JSON

    # Dataset
    train_size: int = 400
    eval_size: int = 100

    # Reproducibility
    seed: int = 42

config = Config()

In [3]:
# Load base file
def load_base():
    bnb = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
    )
    tokenizer = AutoTokenizer.from_pretrained(config.model_name, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
        tokenizer.padding_side = "right"
    model = AutoModelForCausalLM.from_pretrained(
        config.model_name, quantization_config=bnb, device_map="auto", trust_remote_code=True,
    )
    return model, tokenizer

In [ ]:
# Single prompt testing
def test_single_prompt(prompt: str, model, tokenizer, model_name: str):

    # Format with system instruction
    full_prompt = f"{SYSTEM_INSTRUCTION}\n\nCustomer message: {prompt}"

    # Format as Gemma chat
    formatted = (
        f"<start_of_turn>user\n"
        f"{full_prompt}<end_of_turn>\n"
        f"<start_of_turn>model\n"
    )
    
    #pdb.set_trace()

    inputs = tokenizer(formatted, return_tensors="pt").to(model.device)
    
    # Build stop token list
    stop_ids = [tokenizer.eos_token_id]
    eot_id = tokenizer.convert_tokens_to_ids("<end_of_turn>")
    if eot_id is not None and eot_id != tokenizer.unk_token_id:
        stop_ids.append(eot_id)
    
    with torch.no_grad(): # Inferencia
        outputs = model.generate(
            **inputs,
            max_new_tokens=config.max_new_tokens,
            temperature=config.temperature,
            top_p=config.top_p,
            do_sample=config.do_sample,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=stop_ids,
        )
    
    #pdb.set_trace()

    # We slice to remove the echoed input and only decode the new tokens
    # the model generated.  
    generated = tokenizer.decode(
        outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True
    )
    generated = generated.replace("<end_of_turn>", "").replace("<start_of_turn>", "")
    
    return generated.strip()


In [6]:
print("=== Loading base model ===")
base_model, tokenizer = load_base()

# Define your custom test prompts here
test_prompt_1 = "I want a refund for my broken laptop! This is urgent!"
test_prompt_2 = "I can't take it anymore, I will complain to the manager about this terrible service!"

# Test with BASE model
print(f"Prompt: \"{test_prompt_2}\"\n")
print("BASE Model Response:")
base_output = test_single_prompt(test_prompt_2, base_model, tokenizer, "BASE")
print(base_output)
print()
print(f"\n{'-'*70}\n")

lora_model = PeftModel.from_pretrained(base_model, config.output_dir)
lora_model.eval()

print("LoRA Model Response:")
lora_output = test_single_prompt(test_prompt_2, lora_model, tokenizer, "LoRA")
print(lora_output)
print(f"\n{'-'*70}\n")


=== Loading base model ===


[transformers] We found `hidden_act="gelu"` in this Gemma config. This is a legacy value of the official releases but it is meant to target the tanh approximation. Setting `hidden_act="gelu_pytorch_tanh"` instead.


Loading weights:   0%|          | 0/164 [00:00<?, ?it/s]

Prompt: "I can't take it anymore, I will complain to the manager about this terrible service!"

BASE Model Response:
The model is a JSON object with the same structure as the schema.

The model is a JSON object with the same structure as the schema.

The model is a JSON object with the same structure as the schema.

The model is a JSON object with the same structure as the schema.

The model is a JSON object with the same structure as the schema.

The model is a JSON object with the same structure as the schema.

The model is a JSON object with the same structure as the schema.

The model is a JSON object with the same structure as the schema.


----------------------------------------------------------------------

LoRA Model Response:
{"intent": "other", "product": null, "urgency": "high", "needs_human": true, "summary": "Customer expressing strong dissatisfaction with service experience"EndProject model
 Vainqueur model
 lamanguser
model
EndProject
 lamanguser
model
EndProject
 lama

In [7]:
print("=== Loading base model ===")
base_model, tokenizer = load_base()

print("\n=== Evaluating BASE model ===")
base_results = evaluate_model(base_model, tokenizer, config, "Base Model")
print(f"  RAW  — JSON: {base_results[0]:.1%}  |  Schema: {base_results[1]:.1%}")
print(f"  POST — JSON: {base_results[2]:.1%}  |  Schema: {base_results[3]:.1%}")

print(f"\n=== Loading LoRA adapter from: {config.output_dir} ===")
lora_model = PeftModel.from_pretrained(base_model, config.output_dir)
lora_model.eval()

print("\n=== Evaluating LoRA model ===")
lora_results = evaluate_model(lora_model, tokenizer, config, "LoRA Model")
print(f"  RAW  — JSON: {lora_results[0]:.1%}  |  Schema: {lora_results[1]:.1%}")
print(f"  POST — JSON: {lora_results[2]:.1%}  |  Schema: {lora_results[3]:.1%}")

print_evaluation_report(base_results, lora_results)

=== Loading base model ===


Loading weights:   0%|          | 0/164 [00:00<?, ?it/s]


=== Evaluating BASE model ===

Evaluating Base Model...
Processing 20 test cases...

  Processed 5/20 examples
  Processed 10/20 examples
  Processed 15/20 examples
  Processed 20/20 examples
  RAW  — JSON: 0.0%  |  Schema: 0.0%
  POST — JSON: 0.0%  |  Schema: 0.0%

=== Loading LoRA adapter from: ./gemma_lora_json_adapter ===

=== Evaluating LoRA model ===

Evaluating LoRA Model...
Processing 20 test cases...

  Processed 5/20 examples
  Processed 10/20 examples
  Processed 15/20 examples
  Processed 20/20 examples
  RAW  — JSON: 100.0%  |  Schema: 90.0%
  POST — JSON: 100.0%  |  Schema: 90.0%

EVALUATION REPORT

Metric                                   Base Model      LoRA Model     
----------------------------------------------------------------------
  RAW (model output as-is, no cleanup):
    Parseable JSON Rate                           0.0%         100.0%
    Schema Compliance Rate                        0.0%          90.0%
  POST (after JSON extraction):
    Parseable JSON Rat